# Cartographie Sémantique et Analyse des Publications de la FSBM par NLP & Web Scraping

**Projet académique — Faculté des Sciences Ben M'Sick (FSBM), Université Hassan II de Casablanca**

Objectif : constituer un corpus de chercheurs et publications à partir de Google Scholar, normaliser les métadonnées, produire des représentations sémantiques avec **zeroentropy/zembed-1-embedding**, puis explorer le corpus avec un index vectoriel ChromaDB et une recherche par similarité cosinus. Ce carnet examine les artefacts déjà produits : il ne collecte, ne nettoie et ne ré-encode aucune donnée.

## 1. Chaîne de traitement

```text
Google Scholar → scraping prudent → JSON bruts → nettoyage / normalisation / déduplication
→ JSON et Parquet consolidés → zembed-1 → vecteurs à 2 560 dimensions
→ index HNSW ChromaDB (cosinus) → recherche sémantique
```

Le scraping est soumis aux limites d'accès de Google Scholar. Le carnet utilise uniquement les fichiers locaux. L'étape de recherche en direct est désactivée par défaut, car elle charge un modèle volumineux.

## 2. Préparation et chargement des artefacts

Exécuter le carnet depuis la racine du dépôt ou depuis `notebooks/`. La cellule cherche la racine du projet parmi le dossier courant et ses parents. Aucun chemin absolu propre à une machine n'est utilisé.

In [ ]:
from pathlib import Path
import json
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'data/clean/publications.parquet').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Exécuter depuis la racine du projet ou son dossier notebooks/.')

def read_json(relative_path):
    return json.loads((PROJECT_ROOT / relative_path).read_text(encoding='utf-8'))

researchers = pd.DataFrame(read_json('data/clean/researchers.json'))
publications = pd.read_parquet(PROJECT_ROOT / 'data/clean/publications.parquet')
quality = read_json('data/clean/data_quality_report.json')
manifest = read_json('data/embeddings/final_manifest.json')
print('Racine du projet :', PROJECT_ROOT)
print('Chercheurs :', len(researchers), '| Publications uniques :', len(publications))

In [ ]:
overview = pd.Series({
    'Chercheurs uniques': quality['total_unique_researchers'],
    'Publications brutes': quality['raw_publication_records'],
    'Publications uniques': quality['unique_publications'],
    'Éligibles aux embeddings': quality['embedding_eligible'],
    'Exclues des embeddings': quality['embedding_ineligible'],
    'Vecteurs (manifeste)': manifest['publication_count'],
}, name='Effectif')
display(overview.to_frame())
assert len(researchers) == quality['total_unique_researchers']
assert len(publications) == quality['unique_publications']
assert int(publications['embedding_eligible'].sum()) == quality['embedding_eligible']
assert quality['embedding_eligible'] + quality['embedding_ineligible'] == len(publications)

## 3. Chercheurs

Les indicateurs bibliométriques proviennent des profils collectés. Une valeur absente reste manquante : elle n'est pas estimée.

In [ ]:
columns = ['full_name', 'affiliation', 'total_citations', 'h_index', 'i10_index']
display(researchers[columns].head(10).rename(columns={
    'full_name': 'Nom', 'affiliation': 'Affiliation',
    'total_citations': 'Citations', 'h_index': 'Indice h', 'i10_index': 'Indice i10'}))
display(researchers[['total_citations', 'h_index', 'i10_index']].describe().round(1))

## 4. Publications

Une ligne du Parquet représente une publication unique après consolidation. Les listes `researcher_ids` et `article_ids` gardent les liens des notices fusionnées.

In [ ]:
display(publications[['article_id', 'title', 'researcher_name', 'publication_year',
                      'journal', 'citations', 'embedding_eligible']].head(10))
print('Publications uniques :', len(publications))
print('Sans résumé original :', int(publications['abstract'].isna().sum() +
    publications['abstract'].fillna('').astype(str).str.strip().eq('').sum() - publications['abstract'].isna().sum()))
print('Sans résumé nettoyé :', int(publications['abstract_clean'].fillna('').astype(str).str.strip().eq('').sum()))
print('Éligibilité :')
display(publications['embedding_eligible'].value_counts().rename_axis('Éligible').to_frame('Publications'))

In [ ]:
by_year = publications['publication_year'].dropna().astype(int).value_counts().sort_index()
by_researcher = publications.explode('researcher_ids').groupby('researcher_ids').size().sort_values(ascending=False)
name_by_id = researchers.set_index('scholar_id')['full_name'].to_dict()
top_researchers = by_researcher.head(10).rename(index=name_by_id)
display(top_researchers.rename('Publications').to_frame())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_year.plot.bar(ax=axes[0], color='#27648a', width=0.9)
axes[0].set(title='Publications par année', xlabel='Année', ylabel='Nombre')
axes[0].tick_params(axis='x', labelrotation=90)
top_researchers.sort_values().plot.barh(ax=axes[1], color='#63885e')
axes[1].set(title='Dix chercheurs les plus représentés', xlabel='Publications (notices partagées incluses)', ylabel='')
fig.tight_layout()
plt.show()

## 5. Nettoyage et déduplication

Le code de `src/preprocessing/cleaner.py` conserve le titre et le résumé originaux. Le résumé nettoyé applique un décodage HTML, retire les balises, normalise Unicode (NFKC), les espaces et certains caractères de contrôle, puis passe en minuscules sans supprimer les accents. Le titre normalisé sert au rapprochement des doublons. La déduplication privilégie un DOI commun ou un même identifiant d'article pour un chercheur ; sinon elle exige un titre normalisé et une année identiques **avec chevauchement d'auteurs**. Les notices fusionnées conservent leurs identifiants sources.

Une publication est éligible si elle a un titre et un résumé nettoyé d'au moins 20 caractères. Les autres restent dans le Parquet consolidé et sont répertoriées dans `publications_excluded.json`. Le rapport de qualité distingue aussi les doublons fusionnés.

In [ ]:
examples = publications.loc[
    publications['abstract'].fillna('').astype(str).ne(publications['abstract_clean'].fillna('').astype(str)),
    ['title', 'abstract', 'abstract_clean']
].head(3)
display(examples)
print('Doublons fusionnés :', quality['duplicate_publications_merged'])
print('Motifs d’exclusion :', quality['exclusion_reasons'])

## 6. Embeddings finaux

Les vecteurs de documents ont été produits avec `zeroentropy/zembed-1-embedding`, révision épinglée `cf13c81f3274394053d166740294f7eea4586f7a`. Le texte encodé est **`abstract_clean` seul** (`abstract_clean/v1`) ; le titre est une métadonnée de recherche. Le pipeline utilise `encode_document` pour les publications et `encode_query` pour les requêtes. Charger le fichier NPZ ci-dessous **ne charge pas le modèle**.

Si le fichier NPZ n'est pas present dans ce clone, la validation suivante affiche SKIPPED et le carnet continue.

In [ ]:
artifact_path = PROJECT_ROOT / 'data/embeddings/final_publication_embeddings.npz'
ids = None
if not artifact_path.is_file():
    print('SKIPPED: missing data/embeddings/final_publication_embeddings.npz. Install the matching search-artifact package separately to validate vectors.')
else:
    with np.load(artifact_path, allow_pickle=False) as artifact:
        ids = artifact['ids'].tolist()
        vectors = artifact['vectors']
        npz_model = str(artifact['model_name'])
        npz_revision = str(artifact['model_revision'])
        npz_format = str(artifact['document_format'])

    eligible_ids = set(publications.loc[publications['embedding_eligible'], 'article_id'])
    assert vectors.shape == (len(eligible_ids), manifest['embedding_dimension'])
    assert len(ids) == len(set(ids)) and set(ids) == eligible_ids
    assert np.isfinite(vectors).all()
    assert np.all(np.linalg.norm(vectors, axis=1) > 0)
    assert npz_model == manifest['model_name'] == 'zeroentropy/zembed-1-embedding'
    assert npz_revision == manifest['model_revision']
    assert npz_format == manifest['document_format'] == 'abstract_clean/v1'
    display(pd.Series({'Vecteurs': len(ids), 'Dimensions': vectors.shape[1],
                       'Type': str(vectors.dtype), 'Tous finis': bool(np.isfinite(vectors).all()),
                       'Forme d’un vecteur': str(vectors[0].shape)}, name='Validation').to_frame())

## 7. Base vectorielle

L'index final est une collection ChromaDB persistante avec HNSW et la distance cosinus. La vérification ci-dessous ouvre **uniquement SQLite en mode lecture** ; elle ne lance ni ChromaDB ni la construction de l'index. Le même `article_id` relie chaque ligne du Parquet, chaque vecteur NPZ et chaque enregistrement indexé.

Si l'index ChromaDB n'est pas present dans ce clone, la validation suivante affiche SKIPPED et le carnet continue.

In [ ]:
sqlite_file = PROJECT_ROOT / 'data/vector_db_final/chroma.sqlite3'
if not sqlite_file.is_file():
    print('SKIPPED: missing data/vector_db_final/chroma.sqlite3. Install the complete matching ChromaDB directory separately to validate the index.')
elif ids is None:
    print('SKIPPED: the final NPZ is absent, so vector/index ID linkage cannot be validated. Install both search artifacts separately.')
else:
    connection = sqlite3.connect(sqlite_file.resolve().as_uri() + '?mode=ro', uri=True)
    try:
        collection = connection.execute('SELECT name, dimension FROM collections').fetchone()
        metadata = dict(connection.execute('SELECT key, str_value FROM collection_metadata').fetchall())
        indexed_ids = [r[0] for r in connection.execute('SELECT embedding_id FROM embeddings')]
    finally:
        connection.close()
    assert collection == (manifest['collection'], manifest['embedding_dimension'])
    assert metadata['hnsw:space'] == manifest['similarity'] == 'cosine'
    assert metadata['model'] == manifest['model_name']
    assert len(indexed_ids) == len(set(indexed_ids)) == manifest['publication_count']
    assert set(indexed_ids) == set(ids)
    display(pd.Series({'Collection': collection[0], 'Enregistrements': len(indexed_ids),
                       'Métrique': metadata['hnsw:space'], 'IDs concordants': set(indexed_ids) == set(ids)},
                      name='ChromaDB').to_frame())

## 8. Démonstration de recherche sémantique (facultative et coûteuse)

La cellule suivante réutilise `SemanticSearcher`, donc le même index final et l'encodage de requête `encode_query`. **Elle est désactivée par défaut** : l'activer charge le modèle d'environ 4 milliards de paramètres et peut prendre du temps. Aucune publication n'est ré-encodée. Les résultats sont calculés en direct ; aucun classement n'est inventé ou figé dans ce carnet.

In [ ]:
RUN_LIVE_SEARCH = False  # Passer à True seulement pour une démonstration volontaire.
queries = [
    'machine learning for medical diagnosis',
    'natural language processing and text analysis',
    'renewable energy and smart materials',
    'water quality and environmental pollution',
]
if RUN_LIVE_SEARCH:
    import sys
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    from src.search.semantic_search import SemanticSearcher
    searcher = SemanticSearcher()  # Charge le modèle épinglé une seule fois.
    publication_by_id = publications.set_index('article_id')
    for query in queries:
        hits = searcher.search(query, top_k=5)
        rows = []
        for hit in hits:
            article = publication_by_id.loc[hit['article_id']]
            rows.append({'Titre': hit['title'], 'Chercheur': hit['researcher_name'],
                         'Auteurs': ', '.join(article['authors'] or []),
                         'Année': hit['publication_year'],
                         'Similarité cosinus': hit['similarity_score'],
                         'ID': hit['article_id']})
        print('\nRequête :', query)
        display(pd.DataFrame(rows))
else:
    print('Recherche en direct désactivée. Mettre RUN_LIVE_SEARCH = True pour la démonstration.')

## 9. Limite observée de la recherche

Lors de la validation du projet, la requête **« cancer prediction using machine learning »** a retourné surtout des articles généraux sur l'apprentissage automatique plutôt que des travaux spécifiques au cancer. Cette observation concerne le classement pour cette requête, pas la validité des vecteurs. La pertinence dépend notamment de la couverture du corpus, des résumés disponibles, de la formulation de la requête et de la similarité sémantique.

## 10. Conclusion

Le projet fournit une collecte de profils avec reprise et gestion des blocages, un corpus standardisé de **77 chercheurs** et **959 publications uniques**, puis **895 représentations sémantiques** consultables par similarité cosinus. Ses limites comprennent la couverture partielle de Google Scholar, l'indisponibilité fréquente des PDF et références, **64 publications sans embedding**, et une pertinence variable selon les requêtes.

## 11. Reproductibilité et chemins

Artefacts principaux (relatifs à la racine du dépôt) :

- `data/raw/*.json` : sources collectées et conservées ;
- `data/clean/researchers.json`, `publications.json`, `publications.parquet` : corpus final ;
- `data/clean/publications_excluded.json`, `data_quality_report.json` : exclusions et qualité ;
- `data/embeddings/final_publication_embeddings.npz`, `final_manifest.json` : vecteurs et métadonnées ;
- `data/vector_db_final/` : index ChromaDB final.

Commandes à lancer **séparément dans un terminal**, depuis la racine du dépôt :

```powershell
# Nettoyage/consolidation : réécrit les artefacts nettoyés ; inutile pour ce carnet.
python src/preprocessing/merge_raw_data.py

# Génération incrémentale des embeddings : coûteuse ; NE PAS lancer pour le carnet.
python src/embeddings/generate_vectors_and_index.py --incremental-final --batch-size 4 --torch-threads 8

# Recherche ponctuelle : charge le modèle de requête.
python src/search/semantic_search.py "machine learning for medical diagnosis" --top-k 5

# API FastAPI (serveur local).
python -m uvicorn src.api.main:app --host 127.0.0.1 --port 8000
```

Le parcours normal de ce carnet est déterministe, local et en lecture seule. La recherche en direct nécessite une action explicite.